<a href="https://www.kaggle.com/code/ehababoelnaga/anemia-diagnosis?scriptVersionId=177491370" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

#### Install the imortant packages

In [ ]:
# install what you need
%pip install xgboost skimpy

### Import the libraries and moduels needed

In [ ]:
# Libraries for Data manipulation
import pandas as pd
import numpy as np

# Libraries for Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

# Libraries for Stats

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from skimpy import skim

# Libraries for Data preprocessing

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler,LabelEncoder

# Libraries for Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Libraries for Metrics

from sklearn.metrics import balanced_accuracy_score, precision_recall_curve, confusion_matrix,mean_absolute_error,make_scorer, precision_score, f1_score, classification_report,accuracy_score

# Others

from collections import Counter
from tqdm.auto import tqdm
import warnings
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

#### Read the data into dataframe

In [ ]:
data = pd.read_csv('/kaggle/input/anemia-types-classification/diagnosed_cbc_data_v4.csv')
data

## EDA for understanding the data and preprocissing

In [ ]:
skim(data)

In [ ]:
sns.set_style("darkgrid")
features = data.select_dtypes(include = ['int', 'float']).columns.to_list()
colors = sns.color_palette(palette = 'dark', n_colors = len(features))

fig,axs = plt.subplots(nrows = 7, ncols = 2, figsize = (8, 20))
axs = axs.flat

for i,feature in enumerate(features):
    sns.histplot(data, x = feature, stat = 'density', fill = False, color = colors[i], ax = axs[i])
    sns.kdeplot(data, x = feature, fill = True, color = colors[i], ax = axs[i])
    axs[i].set_xlabel("")
    axs[i].set_title(f"{feature}", fontsize = 10, fontweight = 'bold')

fig.tight_layout()
fig.show()

In [ ]:
# As shown in the previous figure, the data needs to be scaled as the features scale is much different
# so we will scale our data
scaler = StandardScaler()
numerical_features = ['WBC', 'LYMp', 'NEUTp', 'LYMn', 'NEUTn', 'RBC', 'HGB', 'HCT', 'MCV', 'MCH', 'MCHC', 'PLT', 'PDW', 'PCT']
data_scaled = data.copy()
data_scaled[numerical_features] = scaler.fit_transform(data_scaled[numerical_features])

In [ ]:
data

In [ ]:
data_scaled

In [ ]:
fig,ax = plt.subplots(figsize = (7,3.5))
ax = sns.countplot(data, y = 'Diagnosis', palette = 'bright', edgecolor = 'black')

for bars in ax.containers:
    ax.bar_label(bars, fontsize = 8, fontweight = 'bold', color = 'black')
    
ax.set_xlabel("")
ax.set_title('Diagnosis', fontsize = 14, fontweight = 'bold', color = 'darkblue')
fig.show()

In [ ]:
fig = go.Figure(data=[go.Pie(labels=data['Diagnosis'], pull=[0.0,0.1],hole=0.5)])
fig.show()

In [ ]:
# As shown our data is not balanced so we need to balance the data
# we need to encode the targer value and split the data first


# Separate features and target variable
X = data.drop(columns=['Diagnosis'])
y = data['Diagnosis']

# Encode target variable
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [ ]:
# Split data into training, validation, and test sets
X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=42)

In [ ]:
# Apply SMOTE with reduced number of neighbors for balancing the data
smote = SMOTE(sampling_strategy='auto', k_neighbors=1, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
# Data after apply SMOTE
fig = go.Figure(data=[go.Pie(labels=y_train_resampled,hole=0.5)])
fig.show()

In [ ]:
corr_matrix = X_train_resampled.corr(method = 'spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype = bool))

fig,ax = plt.subplots(figsize = (12,6.5))
sns.heatmap(corr_matrix, 
            cmap = 'coolwarm', 
            annot = True, 
            annot_kws = {'fontsize':8, 
                         'fontweight':'bold'},
            fmt = '.2f',
            linewidths = 1.2,
            square = True, 
            mask = mask, 
            ax = ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation = 45, ha = 'right')
ax.set_title("Correlation Matrix", fontsize = 12, fontweight = 'bold', color = 'black')
fig.show()

#### Since there is no features greater than 0.9 we will use them all, no need for drop any

## The modeling part

In [ ]:
# Comparing many alogrithms to see what acheives best accuracy
KN = KNeighborsClassifier()
LR = LogisticRegression()
DT = DecisionTreeClassifier()
#SV = SVC()
RF = RandomForestClassifier()
GB = GradientBoostingClassifier()
#LGBM = LGBMClassifier()
XG_B = XGBClassifier()
CB = CatBoostClassifier()
ET = ExtraTreeClassifier()
ETS = ExtraTreesClassifier()
algorithms = [KN,LR,DT,RF,GB,XG_B,CB,ET,ETS]

In [ ]:
def compare_algorithms(X_train, y_train, algorithms, cv=10, scoring='accuracy'):
    """
    Compare the accuracies of multiple machine learning algorithms using cross-validation.

    Parameters:
    - X: Features (input variables) as a numpy array or pandas DataFrame.
    - y: Target variable as a numpy array or pandas Series.
    - algorithms: List of machine learning algorithms to compare.
    - cv: Number of cross-validation folds (default=10).
    - scoring: Evaluation metric to use (default='accuracy').

    Returns:
    - accuracies: Dictionary containing the accuracies of each algorithm.
    """

    accuracies = {}

    for algorithm in algorithms:
        algorithm.fit(X_train, y_train)
        scores = cross_val_score(algorithm, X_val_test, y_val_test, cv=cv, scoring=scoring)
        accuracies[type(algorithm).__name__] = np.mean(scores)

    return accuracies

In [ ]:
def plot_accuracies(accuracies):
    """
    Plot a bar graph of accuracies for each algorithm.

    Parameters:
    - accuracies: Dictionary containing the accuracies of each algorithm.
    """

    algorithms = list(accuracies.keys())
    scores = list(accuracies.values())

    plt.figure(figsize=(10, 6))
    plt.barh(algorithms, scores, color='skyblue')
    plt.xlabel('Accuracy')
    plt.ylabel('Algorithm')
    plt.title('Accuracy of Machine Learning Algorithms')
    plt.xlim(0, 1)  # Set x-axis limits to range from 0 to 1
    plt.show()

In [ ]:
# Compare algorithms and plot accuracies
accuracies = compare_algorithms(X_train_resampled, y_train_resampled, algorithms)


In [ ]:
plot_accuracies(accuracies)

#### Decision Tree classifier has the best score, but it may be overfitting

#### We will train a Decision Tree classifier and try to prevent overfitting if exist

In [ ]:
# fit and train the model
model = DecisionTreeClassifier()
model.fit(X_train_resampled,y_train_resampled)

In [ ]:
# Evaluate The model on training data
y_pred_train = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred_train)
print("Accuracy:", train_accuracy)
print('______________________________________________________')
print("Classification report: \n",classification_report(y_train, y_pred_train))
print('______________________________________________________')
print("\nConfusion Matrix:")
print(confusion_matrix(y_train, y_pred_train))

In [ ]:
# Evaluate The model on training data
y_pred_test = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_test)
print("Accuracy:", test_accuracy)
print('______________________________________________________')
print("Classification report: \n",classification_report(y_test, y_pred_test))
print('______________________________________________________')
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))

In [ ]:
# Evaluate The model on training data
y_pred_val = model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_pred_val)
print("Accuracy:", val_accuracy)
print('______________________________________________________')
print("Classification report: \n",classification_report(y_val, y_pred_val))
print('______________________________________________________')
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_val))

In [ ]:
print(f"Acc Train: {balanced_accuracy_score(y_train, y_pred_train):.4f} | Acc Test: {balanced_accuracy_score(y_test, y_pred_test):.4f}")

In [ ]:
decision_tree = model

In [ ]:

input_data = np.array([[10.1,25.845,77.511,1.88076,	5.14094,3.24,13.4,46.1526,124.1,41.4,33.3,237,14.31251157,0.26028]])  # Example input features
predictions = decision_tree.predict(input_data)
# Interpret predictions
for i, prediction in enumerate(predictions):
    print(f"Input data {i+1}:")
    print("Predicted class:", prediction)